In [ ]:
import pandas as pd
import krippendorff
import numpy as np
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('data/annotation_task/qdr-annotations - main.csv', header=None)
df.rename(columns={0: 'timestamp', 1: 'annotator', 2: 'item', 3:'item_task', 4: 'label'}, inplace=True)
df = df.drop(columns=[5, 6, 7])

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').groupby(['item_task', 'annotator']).tail(1)
df = df[~df['label'].str.contains('Technical error or unable to determine')]

In [ ]:
llm_df = pd.read_csv('data/analysis/initial_data.csv')

llm_df = llm_df[['excerpt_id', 'clarity', 'immediate_relevance', 'specificity', 'attributed_meaning', 'self_reportedness', 'spontaneity', 'intro_context', 'support_rapport', 'elaboration',
       'specifying', 'direct', 'indirect', 'structuring', 'interpreting']].drop_duplicates().dropna()
llm_df['excerpt_id'] = llm_df['excerpt_id'].astype(int)
llm_df = llm_df[llm_df['excerpt_id'].isin(df['item'].unique())]
llm_df = llm_df.sort_values('excerpt_id')

additional_df = pd.read_csv('data/annotation_task/llm_judgements_for_annotation_comparison.csv')
llm_df = llm_df.merge(additional_df[['item_id', 'inclusion_judgement', 'rq_relevance_judgement']], left_on='excerpt_id', right_on='item_id', how='inner')
llm_df = llm_df.rename(columns={'inclusion_judgement': 'inclusion', 'rq_relevance_judgement': 'rq_relevance'})

In [ ]:
tasks = df['item_task'].str.split('_').apply(lambda x: '_'.join(x[1:]) if len(x) > 1 else x[0]).unique()
annotation_dfs = {}
task_dfs = {}

for task in tasks[:-1]:
    print(task)
    task_df = df[df['item_task'].str.contains(rf"^\d+_{task}$", regex=True)].copy().sort_values(['item'])
    task_df['label'] = task_df['label'].str[0]

    item_order = task_df.sort_values('item')['item_task'].unique()

    task_df = task_df.pivot(index='item_task', columns='annotator', values='label')
    task_df = task_df.reindex(sorted(task_df.columns), axis=1).reindex(item_order, axis=0)

    task_df = task_df.astype(float)
    task_df['majority'] = task_df.median(axis=1).round()
    task_df['llm'] = llm_df[task].values
    task_df = task_df.astype(float)

    arr = task_df.to_numpy().T
    arr = arr.astype(float)

    annotation_dfs[task] = arr
    task_dfs[task] = task_df

attributed_meaning
clarity
immediate_relevance
self_reportedness
specificity
spontaneity
rq_relevance
inclusion


In [ ]:
annotators = ['participant1', 'participant2', 'participant3', 'participant4',
       'participant5', 'majority', 'llm']

In [ ]:
print("AGREEMENT AMONG HUMANS")
indices = []

for idx, annotator in enumerate(annotators):
    if annotator in ['participant1', 'participant4', 'participant2', 'participant5', 'participant3']:
        indices.append(idx)

for task, data in annotation_dfs.items():
    print(f"Task: {task}")

    if task == 'inclusion':
        value_domain = [1, 2, 3, 4, 5]
    else:
        value_domain = [1, 2, 3]

    alpha_ordinal = krippendorff.alpha(reliability_data=data[np.array(indices)], level_of_measurement='ordinal', value_domain=value_domain)
    print(f"\tOrdinal Alpha: {alpha_ordinal:.3f}")

AGREEMENT AMONG HUMANS
Task: attributed_meaning
	Ordinal Alpha: 0.750
Task: clarity
	Ordinal Alpha: 0.598
Task: immediate_relevance
	Ordinal Alpha: 0.602
Task: self_reportedness
	Ordinal Alpha: 0.754
Task: specificity
	Ordinal Alpha: 0.732
Task: spontaneity
	Ordinal Alpha: 0.716
Task: rq_relevance
	Ordinal Alpha: 0.714
Task: inclusion
	Ordinal Alpha: 0.754


In [ ]:
print("AGREEMENT BETWEEN MAJORITY AND LLM")

indices = []
for idx, annotator in enumerate(annotators):
    if annotator in ['majority', 'llm']:
        indices.append(idx)

for task, data in annotation_dfs.items():
    print(f"Task: {task}")

    if task == 'inclusion':
        value_domain = [1, 2, 3, 4, 5]
    else:
        value_domain = [1, 2, 3]

    alpha_ordinal = krippendorff.alpha(reliability_data=data[np.array(indices)], level_of_measurement='ordinal', value_domain=value_domain)
    print(f"\tOrdinal Alpha: {alpha_ordinal:.3f}")

AGREEMENT BETWEEN MAJORITY AND LLM
Task: attributed_meaning
	Ordinal Alpha: 0.868
Task: clarity
	Ordinal Alpha: 0.606
Task: immediate_relevance
	Ordinal Alpha: 0.764
Task: self_reportedness
	Ordinal Alpha: 0.679
Task: specificity
	Ordinal Alpha: 0.789
Task: spontaneity
	Ordinal Alpha: 0.797
Task: rq_relevance
	Ordinal Alpha: 0.690
Task: inclusion
	Ordinal Alpha: 0.757


In [ ]:
import ast

task_df = df[df['item_task'].str.contains("interviewer_techniques")].copy().sort_values(['item'])

def to_int_set(x):
    if pd.isna(x):
        return set()
    try:
        vals = ast.literal_eval(x) if isinstance(x, str) else x
    except (ValueError, SyntaxError):
        vals = []
    if not isinstance(vals, (list, tuple, set)):
        return set()

    return {
        int(str(v).strip()[0])
        for v in vals
        if str(v).strip() and str(v).strip()[0].isdigit()
    }

task_df['label'] = task_df['label'].apply(to_int_set)

item_order = task_df.sort_values('item')['item_task'].unique()

piv = task_df.pivot(index='item_task', columns='annotator', values='label')
piv = piv.reindex(sorted(piv.columns), axis=1).reindex(item_order, axis=0)

category_cols = [
    'intro_context',
    'support_rapport',
    'elaboration',
    'specifying',
    'direct',
    'indirect',
    'structuring',
    'interpreting'
 ]

llm_df[category_cols] = llm_df[category_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

llm_df['interviewer_techniques_set'] = llm_df[category_cols].apply(
    lambda row: {i for i, col in enumerate(category_cols, start=1) if row[col] == 1},
    axis=1
)

llm_df = llm_df.sort_values('excerpt_id')

piv['llm'] = llm_df['interviewer_techniques_set'].values

techniques_df = piv

In [ ]:
def jaccard_similarity(list1, list2):

    if pd.isna(list1) or pd.isna(list2):
        return np.nan

    set1, set2 = set(list1), set(list2)

    if not set1 and not set2:
        return 1.0

    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))

    return intersection / union

In [ ]:
techniques_1 = techniques_df.dropna(subset=['participant5', 'participant3'])
techniques_2 = techniques_df.dropna(subset=['participant2', 'participant4', 'participant1'])

In [ ]:
techniques_1['combined'] = techniques_1.apply(lambda row: row['participant3'] | row['participant5'], axis=1)
techniques_2['combined'] = techniques_2.apply(lambda row: row['participant4'] | row['participant2'] | row['participant1'], axis=1)

/var/folders/xk/6w4szy0s6xj0j43rcv1pvyqm0000gn/T/ipykernel_27002/2908506802.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  techniques_1['combined'] = techniques_1.apply(lambda row: row['participant3'] | row['participant5'], axis=1)
/var/folders/xk/6w4szy0s6xj0j43rcv1pvyqm0000gn/T/ipykernel_27002/2908506802.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  techniques_2['combined'] = techniques_2.apply(lambda row: row['participant4'] | row['participant2'] | row['participant1'], axis=1)


In [ ]:
def jaccard_score(annotator1, annotator2):

    if annotator1 in ['participant5', 'participant3'] and annotator2 in ['participant5', 'participant3']:
        return [jaccard_similarity(a1, a2) for a1, a2 in zip(techniques_1[annotator1], techniques_1[annotator2])]
    else:
        return [jaccard_similarity(a1, a2) for a1, a2 in zip(techniques_2[annotator1], techniques_2[annotator2])]

In [ ]:
human_jaccard_scores = jaccard_score('participant5', 'participant3') + jaccard_score('participant4', 'participant2') + jaccard_score('participant4', 'participant1') + jaccard_score('participant2', 'participant1')

llm_jaccard_scores = jaccard_score('participant5', 'llm') + jaccard_score('participant3', 'llm') + jaccard_score('participant4', 'llm') + jaccard_score('participant2', 'llm') + jaccard_score('participant1', 'llm')

print(f"Average Human Jaccard Similarity: {pd.Series(human_jaccard_scores).dropna().mean():.2f}")
print(f"Average LLM Jaccard Similarity: {pd.Series(llm_jaccard_scores).dropna().mean():.2f}")

Average Human Jaccard Similarity: 0.51
Average LLM Jaccard Similarity: 0.50
